In [2]:
%pip install stanza -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import stanza
import pandas as pd
import json
from pathlib import Path

In [ ]:
# ── Parámetros ────────────────────────────────────────────────
CORPUS_PATH   = Path("../data/results/chunks_ciencias_admin_merged.xlsx")   # columna 'texto'

QUERY_LEMMAS = {"orientar", "articular"}

WINDOW_LEFT   = 10                      # tokens a la izquierda
WINDOW_RIGHT  = 10                      # tokens a la derecha

OUTPUT_CSV    = Path("C:\\Users\\karen\\Documents\\HumanidadesDigitales_git\\data\\KWIC\\kwic_orientar_articular.csv")

In [10]:
# ── Carga del corpus ──────────────────────────────────────────
df = pd.read_excel(CORPUS_PATH)
textos = df["texto_chunk"].dropna().tolist() 
len(textos)

# -- Tomar alaetoriamente un subconjunto de textos para acelerar el procesamiento
import random
random.seed(42)
textos = random.sample(textos, min(100, len(textos)))


In [11]:
# ── Pipeline Stanza ───────────────────────────────────────────
# usa GPU si está disponible; si no, CPU
nlp = stanza.Pipeline(
    lang="es",
    processors="tokenize,pos,lemma",
    use_gpu=False,          # cambia a False en partición sin GPU
    tokenize_no_ssplit=True
)

# ── Función de extracción KWIC ────────────────────────────────
def extraer_kwic(doc_id, texto, query_lemmas, win_l, win_r):
    doc = nlp(texto)
    tokens = []
    for sent in doc.sentences:
        for word in sent.words:
            tokens.append({
                "forma": word.text,
                "lema":  word.lemma.lower() if word.lemma else "",
                "pos":   word.upos,
            })

    hits = []
    for i, tok in enumerate(tokens):
        if tok["lema"] in query_lemmas:          
            left  = tokens[max(0, i - win_l) : i]
            right = tokens[i + 1 : i + win_r + 1]
            hits.append({
                "doc_id":       doc_id,
                "lema_buscado": tok["lema"],     # ← columna nueva
                "posicion":     i,
                "forma_hit":    tok["forma"],
                "pos_hit":      tok["pos"],
                "contexto_izq": " ".join(t["forma"] for t in left),
                "hit":          tok["forma"],
                "contexto_der": " ".join(t["forma"] for t in right),
            })
    return hits


# ── Procesamiento del corpus ──────────────────────────────────
todos_hits = []
for doc_id, texto in enumerate(textos):
    hits = extraer_kwic(doc_id, texto, QUERY_LEMMAS, WINDOW_LEFT, WINDOW_RIGHT)
    todos_hits.extend(hits)

# ── Exportación CSV ───────────────────────────────────────────
resultado = pd.DataFrame(todos_hits)
resultado.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")


print(f"Total ocurrencias: {len(todos_hits)}")
print(f"Guardado en: {OUTPUT_CSV}")

2026-06-04 01:55:23 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-06-04 01:55:24 INFO: Downloaded file to C:\Users\karen\AppData\Local\StanfordNLP\stanza\Cache\1.12.0\resources\resources.json
2026-06-04 01:55:24 WARNING: Language es package default expects mwt, which has been added


2026-06-04 01:56:10 INFO: Loading these models for language: es (Spanish):
| Processor | Package           |
---------------------------------
| tokenize  | combined_nocharlm |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

2026-06-04 01:56:10 INFO: Using device: cpu
2026-06-04 01:56:10 INFO: Loading: tokenize
2026-06-04 01:56:29 INFO: Loading: mwt
2026-06-04 01:56:29 INFO: Loading: pos
2026-06-04 01:56:35 INFO: Loading: lemma
2026-06-04 01:56:38 INFO: Done loading processors!


Total ocurrencias: 4
Guardado en: kwic_orientar_articular.csv


In [13]:
print(resultado)

   doc_id lema_buscado  posicion   forma_hit pos_hit  \
0      15     orientar         0   orientado     ADJ   
1      48     orientar       145    orientar    VERB   
2      67     orientar       153    orienten    VERB   
3      89    articular        71  articulado     ADJ   

                                        contexto_izq         hit  \
0                                                      orientado   
1  propuesta ante el Gobierno Nacional el 18 de d...    orientar   
2  estadistas y pensadores de reflexión profunda ...    orienten   
3  cambio sea exitoso debe generar se un sistema ...  articulado   

                                        contexto_der  
0  a sostener , cualificar y potenciar los compro...  
1  la creación y desarrollo de este Instituto de ...  
2  los proyectos que demandan justicia social , p...  
3        y flexible , no solo un ministerio . La ley  
